# Fiche 12 - Exercices : coroutines de base

Executez d'abord les cellules ci-dessous (chargement de la bibliotheque coroutines, puis code du cours : `suspend fun`, `launch`, `delay`, `Job`, puis `async`, `await`, `Deferred`). Completez ensuite les cellules `TODO` avec `Shift+Entree`.

In [ ]:
%use coroutines

In [ ]:
interface Objet {
    val nom: String
    fun description(): String
}
data class Arme(override val nom: String, val degats: Int) : Objet {
    override fun description() = "$nom (+$degats degats)"
}
class Personnage(val nom: String, var pv: Int, var arme: Arme? = null)

suspend fun Personnage.attaquer(cible: Personnage) {
    delay(500)   // temps de preparation de l'attaque
    val degats = arme?.degats ?: 1
    cible.pv = maxOf(0, cible.pv - degats)
    println("$nom attaque ${cible.nom} (-$degats PV)")
}

val heros = Personnage("Aldric", 20, Arme("Epee", 5))
val gobelins = listOf(Personnage("Gobelin 1", 10), Personnage("Gobelin 2", 10))

runBlocking {
    val jobs: List<Job> = gobelins.map { cible -> launch { heros.attaquer(cible) } }
    jobs.forEach { it.join() }   // attend la fin de toutes les attaques
}

gobelins.forEach { println("${it.nom} : ${it.pv} PV") }

In [ ]:
suspend fun Personnage.calculerDegats(): Int {
    delay(300)   // temps de calcul tactique
    return arme?.degats ?: 0
}

runBlocking {
    val allie = Personnage("Elowen", 15, Arme("Dague", 3))

    val degatsHeros: Deferred<Int> = async { heros.calculerDegats() }
    val degatsAllie: Deferred<Int> = async { allie.calculerDegats() }

    println("Degats potentiels cumules : ${degatsHeros.await() + degatsAllie.await()}")
}

## Exercice 1 - Soigner sequentiellement

Ecrivez `suspend fun Personnage.soigner(montant: Int)` qui `delay(300)` (temps de concentration) puis augmente `pv` sans depasser 20 (par exemple avec `minOf(20, ...)`). Dans un `runBlocking`, appelez-la l'une apres l'autre sur deux allies distincts, en mesurant le temps total ecoule avec `System.currentTimeMillis()` avant et apres les deux appels.

In [ ]:
// TODO : ecrivez suspend fun Personnage.soigner(montant: Int) qui delay(300) (temps de
// concentration) puis augmente pv sans depasser 20 (par exemple avec minOf(20, ...))

// runBlocking {
//     val allie1 = Personnage("Allie 1", 10)
//     val allie2 = Personnage("Allie 2", 8)
//     val debut = System.currentTimeMillis()
//     allie1.soigner(5)
//     allie2.soigner(5)
//     println("PV : ${allie1.pv}, ${allie2.pv}")
//     println("Duree : ${System.currentTimeMillis() - debut}ms")
// }

## Exercice 2 - Soigner concurremment

Reprenez le scenario precedent en lancant les deux soins avec `launch` au lieu de les appeler l'un apres l'autre, puis en attendant les deux `Job` avec `join()`. Mesurez a nouveau le temps total ecoule et comparez-le a celui de l'exercice precedent.

In [ ]:
// TODO : reprenez soigner() de l'exercice precedent (recopiez-la si besoin)

// runBlocking {
//     val allie1 = Personnage("Allie 1", 10)
//     val allie2 = Personnage("Allie 2", 8)
//     val debut = System.currentTimeMillis()
//     val job1 = launch { allie1.soigner(5) }
//     val job2 = launch { allie2.soigner(5) }
//     job1.join()
//     job2.join()
//     println("PV : ${allie1.pv}, ${allie2.pv}")
//     println("Duree : ${System.currentTimeMillis() - debut}ms")
// }

## Exercice 3 - Cumuler des resultats avec async

Ecrivez `suspend fun Personnage.calculerDegatsPotentiels(): Int` qui `delay(200)` puis renvoie `arme?.degats ?: 0` (meme principe que `calculerDegats` du cours). Rassemblez le `heros` et deux allies dans une `List<Personnage>`, lancez un `async` par personnage pour calculer concurremment leurs `calculerDegatsPotentiels()`, puis additionnez tous les resultats obtenus.

In [ ]:
// TODO : ecrivez suspend fun Personnage.calculerDegatsPotentiels(): Int qui delay(200)
// puis renvoie arme?.degats ?: 0 (meme principe que calculerDegats du cours)

// runBlocking {
//     val allie1 = Personnage("Allie 1", 15, Arme("Dague", 3))
//     val allie2 = Personnage("Allie 2", 12, Arme("Gourdin", 2))
//     val groupe: List<Personnage> = listOf(heros, allie1, allie2)
//     val deferreds: List<Deferred<Int>> = groupe.map { async { it.calculerDegatsPotentiels() } }
//     val total = deferreds.map { it.await() }.sum()
//     println("Degats potentiels cumules : $total")
// }